# Eval Analysis — flat-scribing pitch sweep (from eval parquets)

Same peak-performance comparison as `peak_perf_analysis`, but every number is read from the
**per-step eval trace parquet attached to each wandb run** (the `eval_*.parquet` on the run's
Files tab) instead of the logged training scalars. Each run's parquet is a full
`(step × env)` rollout trace; this notebook reduces it to **one scalar per metric per agent**
and then feeds those into the *same* `analysis_utils` bar-chart machinery, so the styling
matches `peak_perf_analysis` exactly.

Project `flat_scribing`, tag `iros_ws_v3`: methods `fixed_geo`, `GAS_geo`, `GAS`, `VICES`,
`cholesky` across grasp angles `0°, 15°, 45°` (the `30°` groups have no eval parquet and are
skipped), 5 seeds per cell.

Per-agent reductions from the parquet (each agent = one run = one env slice):

* **Success rate** — fraction of envs that *ever* succeeded: `per_env_ever_success` max per
  env, then mean over envs (matches the training `Episode / Ever success rate`).
* **Mean reward** — per-env total return (`reward` summed over the rollout) meaned over envs
  (matches the scale of the training `Reward / Total reward (mean)`).
* **Average force** — mean of the per-episode drag force `per_env_drag/force` over all envs
  (the raw signal behind the training `drag_performance/force_mean`).
* **Break rate** — from the `termination_cause` column (one categorical code per finished
  episode; see `learning/termination_cause.py`): the fraction of an agent's finished episodes
  whose cause is in the configurable `BREAK_CAUSES` set (default `["peg_break"]`).

Sections **5–8** each build four bar charts (method×angle, angle×method, pooled-by-method,
pooled-by-angle). Section **9** is a curvature box that only renders when the parquets carry a
`curvature_alpha` column (curved-surface tasks): success rate by curvature level, one bar per
method.

`termination_cause` and `curvature_alpha` are published by the env only in traces recorded
**after** that change, so parquets recorded earlier (e.g. the current flat_scribing set) carry
neither — the break section shows empty bars and the curvature box reports "no curvature data"
until the evals are re-run. Downloads are cached as tiny per-run JSONs (the big parquets are
deleted after reduction unless `KEEP_PARQUETS`), so re-running is instant. Figures save as SVG
under `runs/{FOLDER}/plots/eval_analysis/`.

## 1. Imports

In [ ]:
import os
import sys
import json
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# analysis_utils.py sits next to this notebook in iros_workshop_analysis/; learning/ lives at the
# repo root. Put BOTH on sys.path whether the kernel starts here, in data_analysis/, or at the root.
def _ensure_on_path():
    d = os.path.abspath(os.getcwd())
    while True:
        for cand in (d,
                     os.path.join(d, "iros_workshop_analysis"),
                     os.path.join(d, "data_analysis", "iros_workshop_analysis")):
            if os.path.isfile(os.path.join(cand, "analysis_utils.py")) and cand not in sys.path:
                sys.path.insert(0, cand)
        # repo root = the dir that holds learning/termination_cause.py
        if os.path.isfile(os.path.join(d, "learning", "termination_cause.py")) and d not in sys.path:
            sys.path.insert(0, d)
        parent = os.path.dirname(d)
        if parent == d:
            return
        d = parent


_ensure_on_path()
import importlib
import analysis_utils as au   # data loading + styling + plotting (single source of truth)
from learning.termination_cause import CAUSE_NAMES, NAME_TO_CODE  # code<->name for the cause column

# Re-read analysis_utils.py from disk each run so edits take effect WITHOUT a kernel restart.
importlib.reload(au)

## 2. Global parameters

In [ ]:
# --- Data source (wandb). Each run in this project/tag carries a per-step eval trace parquet
# (eval_*.parquet on its Files tab); THAT is what this notebook reads, not the training scalars.
WANDB_ENTITY  = "hur"
WANDB_PROJECT = "flat_scribing"
WANDB_TAG     = "iros_ws_v3"
FOLDER_NAME   = f"{WANDB_PROJECT}_{WANDB_TAG}"

# Re-download + re-reduce every run even if a cached scalar JSON already exists.
FORCE_REFRESH = False
# Keep the (large, ~37 MB) eval parquet on disk after reducing it. False = delete it once the
# per-run scalars are cached, so a full sweep leaves only tiny JSONs behind.
KEEP_PARQUETS = False

# Confidence interval: mean +/- CI_Z * SEM across seeds. 1.96 -> ~95% CI.
CI_Z = 1.96

# --- Termination-cause config. Every finished episode has exactly ONE cause code in the parquet's
# `termination_cause` column (learning/termination_cause.py is the single source of truth). The break
# rate = fraction of an agent's finished episodes whose cause is in this set. Edit this list and just
# re-run the "Process data" cell (the download stores the full per-cause histogram, so changing which
# causes count needs NO re-download). Known causes: success, timeout, traversed, lag, peg_break,
# contact_loss. E.g. ["peg_break", "contact_loss"] to count both hard failures.
BREAK_CAUSES = ["peg_break"]

# --- Metric tags. Names match peak_perf_analysis where they exist so the shared styling (labels, %
# scaling, clips) applies unchanged; BREAK_TAG is local (no display scaling -> shown as a 0-1 rate).
SUCCESS_TAG = "Episode / Success rate"        # per_env_ever_success -> max per env -> mean over envs
REWARD_TAG  = "Reward / Total reward (mean)"  # reward -> sum over rollout per env -> mean over envs
FORCE_TAG   = "drag_performance/force_mean"   # per_env_drag/force -> nanmean over all envs/episodes
BREAK_TAG   = "Break rate"                     # termination_cause in BREAK_CAUSES -> fraction of episodes

# Columns pulled from each parquet (only those that exist are read -> fast, low memory). The last two
# appear only in traces recorded after the env published them (older parquets simply lack them).
PARQUET_COLS = ["env", "step", "per_env_ever_success", "is_success", "reward",
                "per_env_drag/force", "termination_cause", "curvature_alpha"]

# Per-metric config (same structure as peak_perf_analysis). `ci_clip` = (min,max) clips the error-bar
# whiskers; `ylim` sets the y-axis (None = auto).
METRICS = {
    "success": dict(tag=SUCCESS_TAG, ylabel="Success Rate", ci_clip=(0.0, 1.0),  ylim=(0, 1)),
    "reward":  dict(tag=REWARD_TAG,  ylabel="Total Reward", ci_clip=None,        ylim=None),
    "force":   dict(tag=FORCE_TAG,   ylabel="Force (N)",    ci_clip=(0.0, None), ylim=None),
    "break":   dict(tag=BREAK_TAG,   ylabel="Break Rate",   ci_clip=(0.0, 1.0),  ylim=(0, 1)),
}

# Local cache: one tiny JSON per run (keyed by run id) holding the reductions + the raw cause
# histogram + per-curvature success. The eval parquets are downloaded here transiently (and removed
# unless KEEP_PARQUETS). CACHE_KEYS gates re-reduction: a cache missing any of these is re-made.
CACHE_DIR  = os.path.join(au.runs_root(), f"{FOLDER_NAME}_eval_scalars")
CACHE_KEYS = ("success", "reward", "force", "cause_counts", "n_finished", "curvature")

# Output folder: runs/{FOLDER}/plots/eval_analysis/{name}.svg (re-running overwrites in place).
PLOTS_DIR = os.path.join(au.runs_root(), FOLDER_NAME, "plots", "eval_analysis")

# Context object threaded through every figure helper (CI z, output dir, palettes).
STYLE = au.Style(ci_z=CI_Z, xlim=None, plots_dir=PLOTS_DIR)

# Reductions read at eval time: every agent is one logged "row", so mode="eval" -> last value.
MODE = "eval"
SRC  = "eval"

## 3. Load data

In [ ]:
# Download each run's eval parquet, reduce it, and cache the result as a small JSON. The reduction is
# CONFIG-INDEPENDENT (it stores the full per-cause histogram and per-curvature success), so changing
# BREAK_CAUSES later only re-runs "Process data", not the download. Cached runs are skipped unless
# their JSON predates the current CACHE_KEYS (or FORCE_REFRESH). Runs without an eval_*.parquet (e.g.
# the un-evaluated 30 deg groups) are skipped. The big parquet is removed after reduction unless
# KEEP_PARQUETS.
import re

os.makedirs(CACHE_DIR, exist_ok=True)
_AGENT_RE = re.compile(r"agent[_-]?(\d+)", re.IGNORECASE)


def _agent_index(run):
    idx = run.config.get("agent_index") if isinstance(run.config, dict) else None
    if idx is not None:
        return int(idx)
    m = _AGENT_RE.search(run.name or "")
    return int(m.group(1)) if m else 0


def reduce_parquet(path):
    """Collapse one eval trace parquet to the per-run reductions + raw cause/curvature breakdowns."""
    import pyarrow.parquet as pq
    present = set(pq.ParquetFile(path).schema.names)
    df = pd.read_parquet(path, columns=[c for c in PARQUET_COLS if c in present])

    # success: fraction of envs that ever succeeded (fall back to is_success pulses).
    if "per_env_ever_success" in df:
        success = float(df.groupby("env")["per_env_ever_success"].max().mean())
    elif "is_success" in df:
        success = float((df.groupby("env")["is_success"].max() > 0).mean())
    else:
        success = float("nan")

    # reward: per-env total return over the rollout, meaned over envs.
    reward = float(df.groupby("env")["reward"].sum().mean()) if "reward" in df else float("nan")

    # force: mean per-episode drag force over all envs/episodes (NaN except on publish steps).
    force = float(np.nanmean(df["per_env_drag/force"].values)) if "per_env_drag/force" in df else float("nan")

    # termination cause: NaN except on the step an episode finishes -> non-NaN rows = finished episodes.
    cause_counts, n_finished = {}, 0
    if "termination_cause" in df:
        fin = df["termination_cause"].dropna()
        n_finished = int(fin.shape[0])
        cause_counts = {int(k): int(v) for k, v in fin.astype(int).value_counts().items()}

    # curvature: each env has a FIXED alpha; success rate per curvature level (rounded), meaned over
    # the envs at that level. Absent -> None (flat task / pre-change trace).
    curvature = None
    if "curvature_alpha" in df:
        env_alpha = df.groupby("env")["curvature_alpha"].first().round(3)
        env_succ = df.groupby("env")["per_env_ever_success"].max()
        curvature = {}
        for level, grp in env_succ.groupby(env_alpha):
            curvature[f"{float(level):.3f}"] = {"success": float(grp.mean()), "n": int(grp.shape[0])}

    return {"success": success, "reward": reward, "force": force,
            "cause_counts": cause_counts, "n_finished": n_finished, "curvature": curvature}


def download_and_reduce(force=FORCE_REFRESH, keep_parquets=KEEP_PARQUETS):
    import wandb
    api = wandb.Api(timeout=120)
    runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}", filters={"tags": WANDB_TAG})
    print(f"scanning {len(runs)} run(s) in {WANDB_ENTITY}/{WANDB_PROJECT} [tag={WANDB_TAG}]")
    n_done = n_skip = n_no_parquet = 0
    for r in runs:
        cache_path = os.path.join(CACHE_DIR, f"{r.id}.json")
        if os.path.isfile(cache_path) and not force:
            try:
                cached = json.load(open(cache_path))
            except Exception:
                cached = {}
            if all(k in cached for k in CACHE_KEYS):
                n_skip += 1
                continue  # up to date
        pqs = sorted(f for f in r.files()
                     if f.name.startswith("eval_") and f.name.endswith(".parquet"))
        if not pqs:
            n_no_parquet += 1
            continue
        f = pqs[-1]  # latest eval trace if several
        local = f.download(root=os.path.join(CACHE_DIR, "_parquets", r.id),
                           replace=True, exist_ok=True).name
        try:
            reductions = reduce_parquet(local)
        finally:
            if not keep_parquets and os.path.isfile(local):
                os.remove(local)
        rec = {"id": r.id, "name": r.name, "group": r.group,
               "agent": _agent_index(r), **reductions}
        json.dump(rec, open(cache_path, "w"), indent=2)
        n_done += 1
        _cz = sum(reductions["cause_counts"].values())
        print(f"  {r.group:16s} {r.name:22s} succ={reductions['success']:.3f} "
              f"rew={reductions['reward']:8.1f} force={reductions['force']:.3f} "
              f"fin_ep={reductions['n_finished']} curv={'Y' if reductions['curvature'] else '-'}")
    print(f"done: {n_done} reduced, {n_skip} cached, {n_no_parquet} without eval parquet.")


download_and_reduce()

# Collect every cached per-run record.
EVAL_RECORDS = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(CACHE_DIR, "*.json")))]
print(f"loaded {len(EVAL_RECORDS)} cached run record(s)")

## 4. Process data

In [ ]:
# Turn the flat per-run records into the {group: [run, ...]} model analysis_utils expects, where
# each "run" is a single-row eval agent: {tag: (steps=[0], values=[scalar])}. mode="eval" then reads
# that lone value. The break-rate scalar is derived HERE from BREAK_CAUSES (cheap to re-tune).
BREAK_CODES = {NAME_TO_CODE[n] for n in BREAK_CAUSES if n in NAME_TO_CODE}
_unknown = [n for n in BREAK_CAUSES if n not in NAME_TO_CODE]
if _unknown:
    print(f"[warn] BREAK_CAUSES has unknown name(s) {_unknown}; known: {sorted(NAME_TO_CODE)}")


def break_rate(rec):
    """Fraction of a run's finished episodes whose termination cause is in BREAK_CODES (NaN if the
    trace carried no cause column, i.e. no finished episodes recorded)."""
    n = rec.get("n_finished", 0)
    if not n:
        return float("nan")
    cc = rec.get("cause_counts", {}) or {}
    nb = sum(v for k, v in cc.items() if int(k) in BREAK_CODES)
    return nb / n


def _synthetic_run(rec):
    z = np.array([0.0])
    return {
        SUCCESS_TAG: (z, np.array([rec["success"]],  dtype=float)),
        REWARD_TAG:  (z, np.array([rec["reward"]],   dtype=float)),
        FORCE_TAG:   (z, np.array([rec["force"]],    dtype=float)),
        BREAK_TAG:   (z, np.array([break_rate(rec)], dtype=float)),
    }


DATA = {}
for rec in EVAL_RECORDS:
    DATA.setdefault(rec["group"], []).append(_synthetic_run(rec))

COL = au.build_collections(DATA)

# selection_metric is unused in eval mode (each agent is a single row) but the figure API asks for it.
SELECTION_METRIC = SUCCESS_TAG
print("methods:", COL.methods)
print("angles :", COL.angles)
print("cells  :", {f"{m}_{a}": len(COL.by_cell[(m, a)]) for (m, a) in sorted(COL.by_cell)})

# Cause / break-data availability summary.
_n_cause = sum(1 for r in EVAL_RECORDS if r.get("n_finished"))
if _n_cause == 0:
    print("\n[break] NO run carries a `termination_cause` column -> break bars will be empty. "
          "Re-run the evals with the updated trace to populate break rate.")
else:
    _tot = {}
    for r in EVAL_RECORDS:
        for k, v in (r.get("cause_counts") or {}).items():
            _tot[int(k)] = _tot.get(int(k), 0) + v
    print(f"\n[break] {_n_cause}/{len(EVAL_RECORDS)} runs carry causes | counting {BREAK_CAUSES} as break")
    print("[break] cause totals:", {CAUSE_NAMES.get(k, k): v for k, v in sorted(_tot.items())})

## 5. Success rate

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 5. Success rate: x-axis = METHOD, one bar per GRASP ANGLE (colored by angle).
LEGEND_POS = "below"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["success"]
fig = au.figure_bars_method_x_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method x angle ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "success_method_x_angle", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 5. Success rate: x-axis = GRASP ANGLE, one bar per METHOD (colored by method).
LEGEND_POS = "below"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["success"]
fig = au.figure_bars_angle_x_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="Grasp Angle", ylabel=m["ylabel"],
                                    title=None,
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "success_angle_x_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 5. Success rate: aggregated over all runs — one bar per METHOD (pooled over grasp angles).
m = METRICS["success"]
fig = au.figure_bars_by_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method, all angles ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "success_agg_by_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 5. Success rate: aggregated over all runs — one bar per GRASP ANGLE (pooled over methods).
m = METRICS["success"]
fig = au.figure_bars_by_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="grasp angle", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by angle, all methods ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "success_agg_by_angle", STYLE)
plt.show()

## 6. Mean reward

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 6. Mean reward: x-axis = METHOD, one bar per GRASP ANGLE (colored by angle).
LEGEND_POS = "upper left"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["reward"]
fig = au.figure_bars_method_x_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method x angle ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "reward_method_x_angle", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 6. Mean reward: x-axis = GRASP ANGLE, one bar per METHOD (colored by method).
LEGEND_POS = "below"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["reward"]
fig = au.figure_bars_angle_x_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="Grasp Angle", ylabel=m["ylabel"],
                                    title=None,
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "reward_angle_x_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 6. Mean reward: aggregated over all runs — one bar per METHOD (pooled over grasp angles).
m = METRICS["reward"]
fig = au.figure_bars_by_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method, all angles ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "reward_agg_by_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 6. Mean reward: aggregated over all runs — one bar per GRASP ANGLE (pooled over methods).
m = METRICS["reward"]
fig = au.figure_bars_by_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="grasp angle", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by angle, all methods ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "reward_agg_by_angle", STYLE)
plt.show()

## 7. Average force

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 7. Average force: x-axis = METHOD, one bar per GRASP ANGLE (colored by angle).
LEGEND_POS = "below"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["force"]
fig = au.figure_bars_method_x_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method x angle ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "force_method_x_angle", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 7. Average force: x-axis = GRASP ANGLE, one bar per METHOD (colored by method).
LEGEND_POS = "below"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["force"]
fig = au.figure_bars_angle_x_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="Grasp Angle", ylabel=m["ylabel"],
                                    title=None,
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "force_angle_x_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 7. Average force: aggregated over all runs — one bar per METHOD (pooled over grasp angles).
m = METRICS["force"]
fig = au.figure_bars_by_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method, all angles ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "force_agg_by_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 7. Average force: aggregated over all runs — one bar per GRASP ANGLE (pooled over methods).
m = METRICS["force"]
fig = au.figure_bars_by_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="grasp angle", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by angle, all methods ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "force_agg_by_angle", STYLE)
plt.show()

## 8. Break rate

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 8. Break rate: x-axis = METHOD, one bar per GRASP ANGLE (colored by angle).
LEGEND_POS = "below"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["break"]
fig = au.figure_bars_method_x_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method x angle ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "break_method_x_angle", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 8. Break rate: x-axis = GRASP ANGLE, one bar per METHOD (colored by method).
LEGEND_POS = "below"   # "box" | "below" | "upper left" | "upper right" | "lower left" | "lower right"
m = METRICS["break"]
fig = au.figure_bars_angle_x_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="Grasp Angle", ylabel=m["ylabel"],
                                    title=None,
                                    ylim=m["ylim"], ci_clip=m["ci_clip"], legend_pos=LEGEND_POS)
au.save_figure(fig, "break_angle_x_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 8. Break rate: aggregated over all runs — one bar per METHOD (pooled over grasp angles).
m = METRICS["break"]
fig = au.figure_bars_by_method(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="method", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by method, all angles ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "break_agg_by_method", STYLE)
plt.show()

In [ ]:
import importlib; importlib.reload(au)  # hot-reload analysis_utils (edits apply without re-loading data)
# 8. Break rate: aggregated over all runs — one bar per GRASP ANGLE (pooled over methods).
m = METRICS["break"]
fig = au.figure_bars_by_angle(COL, m["tag"], STYLE, mode=MODE, selection_metric=SELECTION_METRIC,
                                    xlabel="grasp angle", ylabel=m["ylabel"],
                                    title=f'{m["ylabel"]} by angle, all methods ({SRC})',
                                    ylim=m["ylim"], ci_clip=m["ci_clip"])
au.save_figure(fig, "break_agg_by_angle", STYLE)
plt.show()

## 9. Success rate by curvature

Only renders when the eval parquets carry a `curvature_alpha` column (curved-surface tasks —
the flat task publishes no alpha, so this stays dormant). x-axis = curvature level (per-env
`alpha`, 0 = flat), one bar per method, height = success rate over the envs at that curvature.

In [ ]:
import importlib; importlib.reload(au)
from collections import defaultdict

# Gather per-(method, curvature level) success across seeds. Curvature success was computed per run
# at reduce time (mean over the envs fixed at each alpha level); here we pool seeds for mean +/- CI.
_curv_records = [r for r in EVAL_RECORDS if r.get("curvature")]
if not _curv_records:
    print("No curvature data (`curvature_alpha`) in these parquets — skipping the curvature plot. "
          "Re-run evals on a curved-surface task to populate it.")
else:
    by_cell = defaultdict(list)   # (method, level_float) -> [per-seed success rate]
    methods, levels = set(), set()
    for r in _curv_records:
        method, _ = au.parse_group(r["group"], default_angle="0")   # method = group minus trailing angle
        methods.add(method)
        for lvl_str, d in r["curvature"].items():
            lvl = round(float(lvl_str), 3)
            levels.add(lvl)
            by_cell[(method, lvl)].append(d["success"])
    methods = au._ordered_methods(methods)
    levels = sorted(levels)

    def _stat(level, method):
        vals = by_cell.get((method, round(float(level), 3)), [])
        if not vals:
            return None
        a = np.array(vals, dtype=float) * 100.0   # show as %
        ci = CI_Z * a.std(ddof=1) / np.sqrt(len(a)) if len(a) > 1 else 0.0
        return float(a.mean()), float(ci)

    fig, ax = plt.subplots(figsize=(au.DEFAULT_FIG_WIDTH, au.DEFAULT_FIG_HEIGHT))
    au.plot_grouped_bars(
        ax, levels, methods, _stat,
        color_fn=STYLE.mcolor, group_name_fn=STYLE.mname, x_name_fn=lambda L: f"{float(L):.2f}",
        xlabel="curvature α", ylabel="Success Rate (%)",
        title=None,  # None (like the angle×method charts) so the "below" legend doesn't overlap it
        ylim=(0, 100), ci_clip=(0, 100), legend_pos="below")
    fig.tight_layout()
    au.save_figure(fig, "success_by_curvature", STYLE)
    plt.show()